# Training and Evaluation in one Notebook for One Model-Database Pair

# To check before running
1. Check class names for your event log in the **p2pencoder.py** ( *{event_log_name}encoder.py* )
2. Check the Axioms in **axiombuilder.py**
3. make sure you have done the declare mining on the event log and have a valid **ltn_rows_path**

In [1]:
event_log_name = "large"
if event_log_name is None:
    raise ValueError("Please set the event_log_name variable to the name of the event log you want to use.")
ltn_rows_path = f"{event_log_name}_ltn_rows.pkl"
print(f"Event log name {event_log_name}")
print(f"LTN Rows path {ltn_rows_path}")

Event log name large
LTN Rows path large_ltn_rows.pkl


In [2]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
# if len(physical_devices) > 0:
#     tf.config.experimental.set_memory_growth(physical_devices[0], True)
#     print("GPU found")
#     print("Memory growth set")
# else:
#     print("No GPU found")

2025-08-07 15:28:14.746797: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-07 15:28:14.757163: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754580494.769596 2335504 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754580494.773437 2335504 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754580494.783184 2335504 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]


In [3]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.largeevaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)

import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'
print(large_leaky_row_classes)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]
Creating Evaluation table
[<class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-10'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-25'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-50'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-100'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-150'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-200'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-250'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-300'>, <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-350'>]


In [4]:
dataset = f"{event_log_name}-0.3-1"
out_dir = PLOT_DIR / f'{event_log_name}_evaluations_both_{arrow.now().format("YYYY-MM-DD-HH-mm-ss")}'
eval_file = out_dir / f'{event_log_name}_fraction_evaluations.pkl'
csv_file = out_dir / f'{event_log_name}_fraction_evaluations.csv'
excel_file = out_dir / f'{event_log_name}_fraction_evaluations.xlsx'
model_folder = r"./.out/models"
db = r"./.out/april.db"

# create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)


Created directory: /home/devashish/Documents/LTNcoder/.out/plots/large_evaluations_both_2025-08-07-15-28-16
Deleted all rows from Evaluation and Model tables.


# Training

In [5]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [6]:
ads = [
        dict(ad=LargeDAE, fit_kwargs=dict(epochs=6, batch_size=100)),
    ] + \
    [
        dict(ad=LEAKY_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100)) for LEAKY_ROW_CLASS 
        in large_leaky_row_classes[:1]
    ] + \
    [
        dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100, epochs_ltn=3))
        for LTN_ROW_CLASS in large_ltn_row_classes
    ]
print(ads)
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


[{'ad': <class 'april.anomalydetection.largeencoder.LargeDAE'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeDAE-Leaky-10'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeLTNFROZEN-10'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeLTNFROZEN-25'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeLTNFROZEN-50'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeLTNFROZEN-100'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeLTNFROZEN-150'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}}, {'ad': <class 'april.anomalydetection.largeencoder.LargeLTNFROZEN-200'>, '

Fitting ADs:   0%|          | 0/11 [00:00<?, ?it/s]

I0000 00:00:1754580497.481442 2335504 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9886 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:3f:00.0, compute capability: 8.9
I0000 00:00:1754580497.482242 2335504 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 30034 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0000:56:00.0, compute capability: 8.9
I0000 00:00:1754580497.482952 2335504 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 30034 MB memory:  -> device: 2, name: NVIDIA L40S, pci bus id: 0000:c3:00.0, compute capability: 8.9
I0000 00:00:1754580497.483506 2335504 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 33610 MB memory:  -> device: 3, name: NVIDIA L40S, pci bus id: 0000:da:00.0, compute capability: 8.9


Epoch 1/6


I0000 00:00:1754580499.606370 2336081 service.cc:152] XLA service 0x7cde0400aa90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1754580499.606419 2336081 service.cc:160]   StreamExecutor device (0): NVIDIA L40S, Compute Capability 8.9
I0000 00:00:1754580499.606427 2336081 service.cc:160]   StreamExecutor device (1): NVIDIA L40S, Compute Capability 8.9
I0000 00:00:1754580499.606432 2336081 service.cc:160]   StreamExecutor device (2): NVIDIA L40S, Compute Capability 8.9
I0000 00:00:1754580499.606436 2336081 service.cc:160]   StreamExecutor device (3): NVIDIA L40S, Compute Capability 8.9
2025-08-07 15:28:19.655551: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1754580499.860580 2336081 cuda_dnn.cc:529] Loaded cuDNN version 90800


...

2025-08-07 15:29:17.555787: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input
Received: inputs=['Tensor(shape=(1, 3842))']
  warnings.warn(msg)


Built forall responded_existence(Activity U, Activity B)
Satisfaction level (test):  0.853496075
Epoch 0
Satisfaction level (train):  0.853643715
Satisfaction level (test):  0.853781819


2025-08-07 15:29:20.005181: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-08-07 15:29:20.167859: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1
Satisfaction level (train):  0.853949547
Satisfaction level (test):  0.854078114
Epoch 2
/home/devashish/Documents/LTNcoder/.out/models/large-0.3-1_largeltnfrozen-10_20250807-152858.128318.keras


NotImplementedError: 

In [ ]:
print(AD) #Evaluator dependso on AD

# Evaluation

In [ ]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [ ]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    print(f"{e} loaded.")
    # print attributes of e
    print(f"e.model_file: {e.model_file}")
    print(f"e.model_name: {e.model_name}")
    print(f"e.eventlog_name: {e.eventlog_name}")
    print(f"e.dataset: {e.dataset}")
    print(f"e.result: {e.result}")
    
    
    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        # print(f"Adding parameters: {e}, {base}, {heuristic}, {strategy}")
        _params.append([e, base, heuristic, strategy])
    
    print(f"{_params} parameters to evaluate.")

    return [_e for p in _params for _e in _evaluate(p)]

In [ ]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

In [ ]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

In [ ]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

orig_ads = [ad['ad'].__name__ for ad in ads if "DAE" in ad['ad'].__name__]
new_ads = [ad['ad'].__name__ for ad in ads if "DAE" not in ad['ad'].__name__]
ads = orig_ads + new_ads

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

In [ ]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')

In [ ]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [ ]:
_filtered_evaluation = evaluation.query(f'ad in {ads} and (strategy == "{Strategy.ATTRIBUTE}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [ ]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad in {orig_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {new_ads})'
                                                )

In [ ]:
df = filtered_evaluation.query('axis == 0')
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df)

In [ ]:
display(df)